# Chakravyuh — Analyzer GRPO Training (Colab Free T4)

**Meta PyTorch OpenEnv Hackathon 2026** — trains a LoRA adapter on **Qwen2.5-3B-Instruct** (fits in free T4, 16 GB VRAM) for the Chakravyuh Behavioral Analyzer.

- Repo: https://github.com/UjjwalPardeshi/Chakravyuh
- Benchmark: `chakravyuh-bench-v0` (n=175) — 5 fraud categories + benign + borderline + multi-turn + adversarial paraphrase
- Scripted baseline: F1 = 0.903, FPR = 9.7%, novel det = 76.5% (the bar LoRA must clear)

**Setup**:
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (or step through cells)
3. Training saves checkpoints to Google Drive → safe to disconnect

**Timing on T4**: 50 episodes ≈ 90-120 min. 200 episodes ≈ 6-8 hours (split across sessions via checkpointing).

## 1. Mount Google Drive (for checkpoint persistence)

Checkpoints save to Drive every 10 steps, so if the Colab session disconnects, the next run resumes from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/chakravyuh'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/analyzer_lora'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint dir:', CHECKPOINT_DIR)

# Check if a previous run left checkpoints we should resume from
RESUME = os.path.isdir(CHECKPOINT_DIR) and any(
    name.startswith('checkpoint-') for name in os.listdir(CHECKPOINT_DIR)
)
print('Resume from existing checkpoint?', RESUME)

## 2. Install dependencies + clone repo

First run: installs take ~3-5 minutes.

In [ ]:
!pip install -q --upgrade "torch>=2.5,<2.6" "transformers>=4.45" "trl>=0.12" \
    "peft>=0.13" "accelerate>=0.33" "bitsandbytes>=0.44" "datasets>=2.19"
!pip install -q --upgrade "unsloth @ git+https://github.com/unslothai/unsloth.git" || echo 'unsloth optional'
!pip install -q "pydantic>=2.6" "numpy>=1.26" "sentence-transformers>=2.7" "wandb"

In [ ]:
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh.git'
if not os.path.exists('/content/Chakravyuh'):
    !git clone $REPO_URL /content/Chakravyuh
else:
    %cd /content/Chakravyuh
    !git pull
os.chdir('/content/Chakravyuh')
!pip install -q -e . ; echo 'installed'

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
import trl, peft, transformers
print('trl:', trl.__version__, '| peft:', peft.__version__, '| transformers:', transformers.__version__)

## 3. Sanity check — build dataset + reward function

Verifies the training pipeline loads correctly without touching the GPU.

In [ ]:
from training.grpo_analyzer import build_training_examples, compute_reward

examples = build_training_examples()
print(f'Built {len(examples)} examples ({sum(1 for e in examples if e.is_scam)} scam / {sum(1 for e in examples if not e.is_scam)} benign)')

ex = examples[0]
good = '{"score": 0.95, "signals": ["urgency", "info_request"], "explanation": "OTP + urgency = scam"}'
bad  = '{"score": 0.1,  "signals": [],                          "explanation": "Looks fine to me"}'
print(f'GOOD completion reward: {compute_reward(good, ex).total:.3f}  (should be ~1.37)')
print(f'BAD  completion reward: {compute_reward(bad, ex).total:.3f}   (should be ~-0.2)')

## 4. Train the LoRA (Qwen2.5-3B + GRPO)

- **Model**: Qwen2.5-3B-Instruct (4-bit quantized, fits T4)
- **Episodes**: 50 for first smoke run — bump to 200 once you know it converges
- **Output**: LoRA adapter saved to Google Drive every 10 steps (resumable)
- **Batch size 1 + grad accum 4** — tight VRAM safety margin on T4

If you hit OOM: drop to `Qwen/Qwen2.5-1.5B-Instruct` or `Qwen/Qwen2.5-0.5B-Instruct`.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
EPISODES = 50           # smoke run. Bump to 200 after verifying convergence.

# Training output goes directly to Drive so checkpoints survive disconnect.
# Resume-from-checkpoint is enabled by --resume (reads from $CHECKPOINT_DIR).
RESUME_FLAG = '--resume' if RESUME else ''

!python -m training.grpo_analyzer \
    --model $MODEL_NAME \
    --episodes $EPISODES \
    --output $CHECKPOINT_DIR \
    --lora-rank 16 \
    --batch-size 1 \
    --grad-accum 4 \
    --num-generations 4 \
    --no-wandb \
    $RESUME_FLAG

## 5. Evaluate the trained adapter on chakravyuh-bench-v0

Compare trained LoRA to the ScriptedAnalyzer baseline (F1 = 0.903).

In [ ]:
from chakravyuh_env.agents.llm_analyzer import LLMAnalyzer
from eval.mode_c_real_cases import load_dataset, run_eval, aggregate, per_category_breakdown, per_difficulty_breakdown, DEFAULT_DATASET

analyzer = LLMAnalyzer(
    model_name=MODEL_NAME,
    lora_path=CHECKPOINT_DIR,
    use_unsloth=True,
    load_in_4bit=True,
)
analyzer.load()

data = load_dataset(DEFAULT_DATASET)
results = run_eval(analyzer, data, threshold=0.5)
metrics = aggregate(results)
print(f'=== LoRA results vs scripted baseline (F1=0.903) ===')
print(f'  Detection: {metrics.detection_rate:.1%}  (scripted: 84.0%)')
print(f'  Precision: {metrics.precision:.1%}  (scripted: 97.6%)')
print(f'  FPR:       {metrics.false_positive_rate:.1%}  (scripted: 9.7%)')
print(f'  F1:        {metrics.f1:.3f}  (scripted: 0.903)')

print()
by_diff = per_difficulty_breakdown(results)
if 'novel' in by_diff:
    print(f'Novel detection: {by_diff["novel"].detection_rate:.1%}  (scripted: 76.5%)')

## 6. Save final LoRA + eval results

Final adapter is already on Drive (training output_dir was Drive). This cell also saves the eval metrics alongside.

In [ ]:
import json

lora_eval = {
    'model': MODEL_NAME,
    'lora_path': CHECKPOINT_DIR,
    'episodes': EPISODES,
    'n_eval': metrics.n,
    'detection_rate': metrics.detection_rate,
    'false_positive_rate': metrics.false_positive_rate,
    'precision': metrics.precision,
    'recall': metrics.recall,
    'f1': metrics.f1,
    'accuracy': metrics.accuracy,
    'by_category': {k: v.__dict__ for k, v in per_category_breakdown(results).items()},
    'by_difficulty': {k: v.__dict__ for k, v in per_difficulty_breakdown(results).items()},
}
with open(f'{CHECKPOINT_DIR}/eval_results.json', 'w') as f:
    json.dump(lora_eval, f, indent=2)
print(f'Saved eval_results.json to {CHECKPOINT_DIR}')
print(f'Download LoRA adapter from {CHECKPOINT_DIR} for local use.')

## Next steps

1. **If 50-episode smoke run beats scripted baseline** → change `EPISODES = 200` and re-run (6-8 hrs, possibly across multiple sessions thanks to checkpointing).
2. **If it doesn't converge** → inspect `eval_results.json`, adjust reward weights in `training/grpo_analyzer.py` (compute_reward function).
3. **If you have RunPod budget** → swap `MODEL_NAME` to `Qwen/Qwen2.5-7B-Instruct`, bump `batch-size 4`, run on A100.
4. **Push to HF Hub**: `huggingface-cli upload chakravyuh/analyzer-lora $CHECKPOINT_DIR`

Chakravyuh · Meta PyTorch OpenEnv Hackathon · April 2026